In [49]:
import nltk
import re
import utils
from collections import Counter

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.data.path.append('.')

[nltk_data] Downloading package punkt to /Users/dyh/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/dyh/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [74]:
class SpecialTokens:
    """ 
    Class of special tokens
    """
    def __init__(self, start_token = "<s>", end_token = "<e>", unknown_token = "<unk>"):
        self.start_token = start_token
        self.end_token = end_token
        self.unknown_token = unknown_token

class NGramModel():
    """
    This class holds your n-gram model and all its parameters, which include:

      - previous_n_gram: the last n-gram that appeared prior to the current one
      - n-gram_counts: a dictionary specifying the context words
    """
    def __init__(self, n_gram_counts,
               n_plus1_gram_counts, vocabulary, special_tokens, k=1):
        self.n_gram_counts = n_gram_counts
        self.n_plus1_gram_counts = n_plus1_gram_counts
        self.vocabulary = vocabulary
        self.special_tokens = special_tokens
        self.k = k

In [2]:
#@title Question 1

def preprocess_data(filename, count_threshold, special_tokens,
                    sample_delimiter='\n', split_ratio=0.8):
    """
    Ungraded: You do not need to change this function.

    Preprocess data, i.e.,
        - Find tokens that appear at least N times in the training data.
        - Replace tokens that appear less than N times by "<unk>" .
    Args:
        count_threshold: Words whose count is less than this are
                      treated as unknown.

    Returns:
        training_data = list of lists denoting tokenized sentence. This looks like
                        the following:
 
                        [ ["this", "<unk>", "example"], 
                          ["another", "sentence", "<unk>", "right"],
                         ...
                        ] 
        test_data = Same format as above.
        vocabulary = list of vocabulary words. This looks like the following:

                        ["vocab-word-1", "vocab-word-2", etc.]
    """

    # Create sentences and tokenize the data to create a list of strings. 
    tokenized_data = read_and_tokenize_sentences(filename, sample_delimiter)

    # Create the training / test splits
    train_size = int(len(tokenized_data) * split_ratio)
    train_data = tokenized_data[0:train_size]
    test_data = tokenized_data[train_size:]

    # Get the closed vocabulary using the train data
    vocabulary = get_words_with_nplus_frequency(train_data, count_threshold)

    # For the train data, replace less common words with unknown token
    train_data_replaced = replace_oov_words_by_unk(
        train_data, vocabulary, unknown_token = special_tokens.unknown_token)

    # For the test data, replace less common words with "<unk>"
    test_data_replaced = replace_oov_words_by_unk(
        test_data, vocabulary, unknown_token = special_tokens.unknown_token)

    return train_data_replaced, test_data_replaced, vocabulary

def preprocess_data_test():
    """
    Ungraded: You can use this function to test out preprocess_data. 
    """
    tmp_train = "the sky is blue.\nleaves are green.\nsmell all the roses."
    tmp_test = "roses are red."

    with open('tmp_data.txt', 'w') as f:
      f.write(str(tmp_train) + '\n')
      f.write(str(tmp_test) + '\n')

    special_tokens = SpecialTokens()
    count_threshold = 1

    tmp_train_repl, tmp_test_repl, tmp_vocab = preprocess_data(
        "tmp_data.txt", count_threshold, special_tokens, split_ratio = 0.75)

    assert tmp_test_repl == [['roses', 'are', '<unk>', '.']] or \
      tmp_test_repl == [[special_tokens.start_token, 
                         'roses', 'are', '<unk>', 
                         special_tokens.end_token]] or \
      tmp_test_repl == [[special_tokens.start_token, 
                         'roses', 'are', '<unk>', '.',
                         special_tokens.end_token]], \
      print("tmp_test_repl is not correct")

    assert tmp_train_repl == [['the', 'sky', 'is', 'blue', '.'],
                              ['leaves', 'are', 'green', '.'],
                              ['smell', 'all', 'the', 'roses', '.']] or \
           tmp_train_repl == [[special_tokens.start_token, 
                               'the', 'sky', 'is', 'blue', 
                               special_tokens.end_token],
                              [special_tokens.start_token, 
                               'leaves', 'are', 'green', 
                               special_tokens.end_token],
                              [special_tokens.start_token, 
                               'smell', 'all', 'the', 'roses', 
                               special_tokens.end_token]] or \
           tmp_train_repl == [[special_tokens.start_token, 
                               'the', 'sky', 'is', 'blue', '.',
                               special_tokens.end_token],
                              [special_tokens.start_token, 
                               'leaves', 'are', 'green', '.',
                               special_tokens.end_token],
                              [special_tokens.start_token, 
                               'smell', 'all', 'the', 'roses', '.',
                               special_tokens.end_token]], \
      print("tmp_train_repl is not correct")

    print("\033[92m Successful test")

    return 

In [79]:
#@title Q1.1 Read / Tokenize Data from Sentences

def read_and_tokenize_sentences(filename, sample_delimiter="\n"):
    """
    Args:
        - filename = (e.g., "en_US.twitter.txt")
        - sample_delimiter = delimits each sample (i.e., each tweet)

    Example usage: 
       $ read_and_tokenize_sentences(sentences) 

       [['sky', 'is', 'blue', '.'],
        ['leaves', 'are', 'green'],
        ['roses', 'are', 'red', '.']]A

    You can use nltk's tokenize function here.

       nltk.word_tokenize(sentence)
    """
    # <YOUR-CODE-HERE>
    with open(filename, "r", encoding="utf-8") as f:
        corpus = f.read().lower()
    # corpus = re.sub(r"[^a-zA-Z0-9.?! ]+", "", corpus)
    
    sentences = corpus.split(sample_delimiter)

    tokenized_sentences = []
    
    for sentence in sentences:
        if sentence.strip():
            tokens = nltk.word_tokenize(sentence)
            tokenized_sentences.append(tokens)
        
    return tokenized_sentences

def get_words_with_nplus_frequency(train_data, count_threshold):
    # <YOUR-CODE-HERE>
    word_freq = Counter()
    for tweet in train_data:
        word_freq.update(tweet)

    vocabulary = [word for word, freq in word_freq.items() if freq >= count_threshold]
    
    return vocabulary

#@title Q1.2 Replace OOV Words with Special Token

def replace_oov_words_by_unk(data, vocabulary, unknown_token="<unk>"):
    # <YOUR-CODE-HERE>
    data_replaced = []

    vocab_set = set(vocabulary)
    
    for tweet in data:
        new_tweet = []
        for word in tweet:
            if word in vocab_set:
                new_tweet.append(word)
            else:
                new_tweet.append(unknown_token)
        data_replaced.append(new_tweet)
    return data_replaced

In [ ]:
read_and_tokenize_sentences("data/en_US.twitter.txt", sample_delimiter="\n")

In [83]:
tokenized_data = read_and_tokenize_sentences("data/en_US.twitter.txt", sample_delimiter="\n")
train_size = int(len(tokenized_data) * 0.8)
train_data = tokenized_data[0:train_size]
vocabulary = get_words_with_nplus_frequency(train_data, count_threshold = 7000)
train_data_replaced = replace_oov_words_by_unk(train_data, vocabulary, "<unk>")


In [85]:

import nltk
import utils
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.data.path.append('.')

class SpecialTokens:
    """ 
    Class of special tokens
    """
    def __init__(self, start_token = "<s>", end_token = "<e>", unknown_token = "<unk>"):
        self.start_token = start_token
        self.end_token = end_token
        self.unknown_token = unknown_token

class NGramModel():
    """
    This class holds your n-gram model and all its parameters, which include:

      - previous_n_gram: the last n-gram that appeared prior to the current one
      - n-gram_counts: a dictionary specifying the context words
    """
    def __init__(self, n_gram_counts,
               n_plus1_gram_counts, vocabulary, special_tokens, k=1):
        self.n_gram_counts = n_gram_counts
        self.n_plus1_gram_counts = n_plus1_gram_counts
        self.vocabulary = vocabulary
        self.special_tokens = special_tokens
        self.k = k

#@title Question 1

def preprocess_data(filename, count_threshold, special_tokens,
                    sample_delimiter='\n', split_ratio=0.8):
    """
    Ungraded: You do not need to change this function.

    Preprocess data, i.e.,
        - Find tokens that appear at least N times in the training data.
        - Replace tokens that appear less than N times by "<unk>" .
    Args:
        count_threshold: Words whose count is less than this are
                      treated as unknown.

    Returns:
        training_data = list of lists denoting tokenized sentence. This looks like
                        the following:
 
                        [ ["this", "<unk>", "example"], 
                          ["another", "sentence", "<unk>", "right"],
                         ...
                        ] 
        test_data = Same format as above.
        vocabulary = list of vocabulary words. This looks like the following:

                        ["vocab-word-1", "vocab-word-2", etc.]
    """

    # Create sentences and tokenize the data to create a list of strings. 
    tokenized_data = read_and_tokenize_sentences(filename, sample_delimiter)

    # Create the training / test splits
    train_size = int(len(tokenized_data) * split_ratio)
    train_data = tokenized_data[0:train_size]
    test_data = tokenized_data[train_size:]

    # Get the closed vocabulary using the train data
    vocabulary = get_words_with_nplus_frequency(train_data, count_threshold)

    # For the train data, replace less common words with unknown token
    train_data_replaced = replace_oov_words_by_unk(
        train_data, vocabulary, unknown_token = special_tokens.unknown_token)

    # For the test data, replace less common words with "<unk>"
    test_data_replaced = replace_oov_words_by_unk(
        test_data, vocabulary, unknown_token = special_tokens.unknown_token)

    return train_data_replaced, test_data_replaced, vocabulary

def preprocess_data_test():
    """
    Ungraded: You can use this function to test out preprocess_data. 
    """
    tmp_train = "the sky is blue.\nleaves are green.\nsmell all the roses."
    tmp_test = "roses are red."

    with open('tmp_data.txt', 'w') as f:
      f.write(str(tmp_train) + '\n')
      f.write(str(tmp_test) + '\n')

    special_tokens = SpecialTokens()
    count_threshold = 1

    tmp_train_repl, tmp_test_repl, tmp_vocab = preprocess_data(
        "tmp_data.txt", count_threshold, special_tokens, split_ratio = 0.75)

    assert tmp_test_repl == [['roses', 'are', '<unk>', '.']] or \
      tmp_test_repl == [[special_tokens.start_token, 
                         'roses', 'are', '<unk>', 
                         special_tokens.end_token]] or \
      tmp_test_repl == [[special_tokens.start_token, 
                         'roses', 'are', '<unk>', '.',
                         special_tokens.end_token]], \
      print("tmp_test_repl is not correct")

    assert tmp_train_repl == [['the', 'sky', 'is', 'blue', '.'],
                              ['leaves', 'are', 'green', '.'],
                              ['smell', 'all', 'the', 'roses', '.']] or \
           tmp_train_repl == [[special_tokens.start_token, 
                               'the', 'sky', 'is', 'blue', 
                               special_tokens.end_token],
                              [special_tokens.start_token, 
                               'leaves', 'are', 'green', 
                               special_tokens.end_token],
                              [special_tokens.start_token, 
                               'smell', 'all', 'the', 'roses', 
                               special_tokens.end_token]] or \
           tmp_train_repl == [[special_tokens.start_token, 
                               'the', 'sky', 'is', 'blue', '.',
                               special_tokens.end_token],
                              [special_tokens.start_token, 
                               'leaves', 'are', 'green', '.',
                               special_tokens.end_token],
                              [special_tokens.start_token, 
                               'smell', 'all', 'the', 'roses', '.',
                               special_tokens.end_token]], \
      print("tmp_train_repl is not correct")

    print("\033[92m Successful test")

    return 

#@title Q1.1 Read / Tokenize Data from Sentences

def read_and_tokenize_sentences(filename, sample_delimiter="\n"):
    """
    Args:
        - filename = (e.g., "en_US.twitter.txt")
        - sample_delimiter = delimits each sample (i.e., each tweet)

    Example usage: 
       $ read_and_tokenize_sentences(sentences) 

       [['sky', 'is', 'blue', '.'],
        ['leaves', 'are', 'green'],
        ['roses', 'are', 'red', '.']]A

    You can use nltk's tokenize function here.

       nltk.word_tokenize(sentence)
    """
    # <YOUR-CODE-HERE>
    with open(filename, "r", encoding="utf-8") as f:
        corpus = f.read().lower()
    # corpus = re.sub(r"[^a-zA-Z0-9.?! ]+", "", corpus)
    
    sentences = corpus.split(sample_delimiter)

    tokenized_sentences = []
    
    for sentence in sentences:
        if sentence.strip():
            tokens = nltk.word_tokenize(sentence)
            tokenized_sentences.append(tokens)
        
    return tokenized_sentences

def get_words_with_nplus_frequency(train_data, count_threshold):
    # <YOUR-CODE-HERE>
    word_freq = Counter()
    for tweet in train_data:
        word_freq.update(tweet)

    vocabulary = [word for word, freq in word_freq.items() if freq >= count_threshold]
    
    return vocabulary

#@title Q1.2 Replace OOV Words with Special Token

def replace_oov_words_by_unk(data, vocabulary, unknown_token="<unk>"):
    # <YOUR-CODE-HERE>
    data_replaced = []

    vocab_set = set(vocabulary)
    
    for tweet in data:
        new_tweet = []
        for word in tweet:
            if word in vocab_set:
                new_tweet.append(word)
            else:
                new_tweet.append(unknown_token)
        data_replaced.append(new_tweet)
    return data_replaced

special_tokens = SpecialTokens()
count_threshold = 10
train_data_replaced, test_data_replaced, vocabulary = preprocess_data(
        "data/en_US.twitter.txt", count_threshold, special_tokens)

preprocess_data_test()

[nltk_data] Downloading package punkt to /Users/dyh/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/dyh/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


 Successful test


In [92]:
#@title Q2 Count N-Grams

def count_n_grams(data, n, special_tokens):
    """
    Count all n-grams in the data

    Args:
        data: List of lists of words
        n: Number of words in a sequence
        special_tokens: A structure that contains:
          - start_token = "<s>"
          - end_token = "<e>"
          - unknown_token = "unk"

    Returns:
        A dictionary that maps a tuple of n-words to its frequency
    """

    # Initialize dictionary of n-grams and their counts
    n_grams = {}
    # <YOUR-CODE-HERE>
    for tweet in data:
        new_tweet = [special_tokens.start_token] * (n - 1) + tweet + [special_tokens.end_token]
        
        for i in range(len(new_tweet) - n + 1):
            gram = tuple(new_tweet[i: i + n])
            n_grams[gram] = n_grams.get(gram, 0) + 1
    
    return n_grams

def count_n_grams_test():
    """
    Ungraded: You can use this function to test out count_n_grams. 
    """
    tmp_data = "i like a cat\nthis dog is like a cat"
    with open('tmp_data.txt', 'w') as f:
      f.write(tmp_data + '\n')

    sentences, _, _ = preprocess_data(
        "tmp_data.txt", 0, SpecialTokens(), split_ratio = 1.0)

    received = count_n_grams(sentences, 2, SpecialTokens())
    expected = { ('<s>', 'i'): 1,
      ('i', 'like'): 1, ('like', 'a'): 2, ('a', 'cat'): 2, ('cat', '<e>'): 2,
      ('<s>', 'this'): 1, ('this', 'dog'): 1, ('dog', 'is'): 1, ('is', 'like'): 1}

    assert received == expected, print("Received: \n", received, 
                                       "\n\nExpected: \n", expected)

    print("\033[92m Successful test")

    return
	


IndentationError: unindent does not match any outer indentation level (<string>, line 68)

In [90]:
count_n_grams_test()

 Successful test


In [95]:
def make_probability_matrix(n_plus1_gram_counts, vocabulary, k):

    count_matrix = utils.make_count_matrix(

        n_plus1_gram_counts,

        vocabulary

    )

    count_matrix += k

    prob_matrix = count_matrix.div(

        count_matrix.sum(axis=1),

        axis=0

    )

    return prob_matrix

sentences = [

    ['i', 'like', 'a', 'cat'],

    ['this', 'dog', 'is', 'like', 'a', 'cat']

]

unique_words = list(set(sentences[0] + sentences[1]))

bigram_counts = count_n_grams(

    sentences,

    2,

    SpecialTokens()

)

print("Bigram Counts")

display(utils.make_count_matrix(bigram_counts, unique_words))

print("Bigram Probabilities")

display(make_probability_matrix(bigram_counts, unique_words, k=1))

Bigram Counts


,cat,a,i,is,dog,like,this,<e>,<unk>
"(dog,)",0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
"(cat,)",0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0
"(like,)",0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"(a,)",2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"(<s>,)",0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
"(i,)",0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
"(this,)",0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
"(is,)",0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


Bigram Probabilities


,cat,a,i,is,dog,like,this,<e>,<unk>
"(dog,)",0.100000,0.100000,0.100000,0.200000,0.100000,0.100000,0.100000,0.100000,0.100000
"(cat,)",0.090909,0.090909,0.090909,0.090909,0.090909,0.090909,0.090909,0.272727,0.090909
"(like,)",0.090909,0.272727,0.090909,0.090909,0.090909,0.090909,0.090909,0.090909,0.090909
"(a,)",0.272727,0.090909,0.090909,0.090909,0.090909,0.090909,0.090909,0.090909,0.090909
"(<s>,)",0.090909,0.090909,0.181818,0.090909,0.090909,0.090909,0.181818,0.090909,0.090909
"(i,)",0.100000,0.100000,0.100000,0.100000,0.100000,0.200000,0.100000,0.100000,0.100000
"(this,)",0.100000,0.100000,0.100000,0.100000,0.200000,0.100000,0.100000,0.100000,0.100000
"(is,)",0.100000,0.100000,0.100000,0.100000,0.100000,0.200000,0.100000,0.100000,0.100000


In [115]:
#@title Q3 Estimate the Probabilities

def estimate_probabilities(context_tokens, ngram_model):
    """
    Estimate the probabilities of a next word using the n-gram counts with k-smoothing

    Args:
        word: next word
        previous_n_gram: A sequence of words of length n
        ngram_model: a structure that contains:
            - n_gram_counts: Dictionary of counts of n-grams
            - n_plus1_gram_counts: Dictionary of counts of (n+1)-grams
            - vocabulary_size: number of words
            - k: positive constant, smoothing parameter

    Returns:
        A dictionary mapping from next words to probability
    """
    probabilities = {}
    # <YOUR-CODE-HERE>
    previous_n_gram_count = ngram_model.n_gram_counts.get(tuple(context_tokens), 0)

    # Include special tokens and remove duplicates
    vocabulary = ngram_model.vocabulary + ["<e>", "<unk>"]


    vocabulary_size = len(vocabulary)
    k = ngram_model.k    
    denominator = previous_n_gram_count + k * vocabulary_size  

    for word in vocabulary:
        n_plus1_gram = tuple(context_tokens) + (word,)
        count = ngram_model.n_plus1_gram_counts.get(n_plus1_gram, 0)
        numerator = count + k      
        probabilities[word] = numerator / denominator
    
    return probabilities


def estimate_probabilities_test():
    """
    Ungraded: You can use this function to test out estimate_probabilities. 
    """

    tmp_data = "i like a cat\nthis dog is like a cat"
    with open('tmp_data.txt', 'w') as f:
      f.write(tmp_data + '\n')

    sentences, _, vocabulary = preprocess_data(
        "tmp_data.txt", 0, SpecialTokens(), split_ratio = 1.0)

    # unique_words = list(set(sentences[0] + sentences[1]))
    unigram_counts = count_n_grams(sentences, 1, SpecialTokens())
    bigram_counts = count_n_grams(sentences, 2, SpecialTokens())

    ngram_model = NGramModel(unigram_counts, bigram_counts, vocabulary,
                             SpecialTokens(), k=1)
    
    expected = {'i': 0.09090909090909091, 'like': 0.09090909090909091, 
                'a': 0.09090909090909091, 'cat': 0.2727272727272727, 
                'this': 0.09090909090909091, 'dog': 0.09090909090909091, 
                'is': 0.09090909090909091, '<e>': 0.09090909090909091, 
                '<unk>': 0.09090909090909091}

    assert estimate_probabilities(["a"], ngram_model) == expected, \
      print("estimate_probabilities is not correct")

    
    print("\033[92m Successful test")

In [116]:
estimate_probabilities_test()

 Successful test


In [119]:
#@title Q4 Inference

def predict_next_word(partial_sentence, ngram_model):
    """
    Predicts the next word in a partial sentence using the provided n-gram model.

    Args:
        partial_sentence: The partial sentence as a string.
        ngram_model: An instance of the NGramModel class.

    Returns:
        The predicted next word (string) or None if no prediction can be made.
    """
    # <YOUR-CODE-HERE>
    context_tokens = nltk.word_tokenize(partial_sentence.lower())

    n = len(next(iter(ngram_model.n_gram_counts)))
    context_tokens = context_tokens[-n:]
      
    probabilities = estimate_probabilities(context_tokens, ngram_model)

    predicted_word = None
    max_prob = 0
    
    for word, prob in probabilities.items():
        if prob > max_prob:
            predicted_word = word
            max_prob = prob
            
    return predicted_word

In [120]:

# n=2
unigram_counts = count_n_grams(train_data_replaced, 1, special_tokens)
bigram_counts = count_n_grams(train_data_replaced, 2, special_tokens)


ngram_model = NGramModel(unigram_counts, bigram_counts, vocabulary,
                        special_tokens, k=1)

partial_sentence = "i love"  # Example partial sentence
predicted_word = predict_next_word(partial_sentence, ngram_model)
print(f"The predicted next word for '{partial_sentence}' is: {predicted_word}")


The predicted next word for 'i love' is: you


In [1]:
import math
# @title Q5 Extra Credit

class StyleGram:

    def __init__(self, style_files):
        """
        Train one bigram language model for each writing style.
        All preprocessing and training are completed here.
        """
        self.style_files = style_files
        self.special_tokens = SpecialTokens()
        self.models = []

        # Read and tokenize all style datasets
        self.style_data = []

        for filename in style_files:
            sentences = read_and_tokenize_sentences(filename)
            self.style_data.append(sentences)

        # Build a shared vocabulary across all styles
        word_counts = Counter()

        for sentences in self.style_data:
            for sentence in sentences:
                word_counts.update(sentence)

        self.vocabulary = [
            word for word, count in word_counts.items()
            if count >= 2
        ]

        self.vocab_set = set(self.vocabulary)

        # Train a separate bigram model for each style
        self.style_token_counts = []

        for sentences in self.style_data:

            processed_data = replace_oov_words_by_unk(
                sentences,
                self.vocabulary,
                self.special_tokens.unknown_token
            )

            unigram_counts = count_n_grams(
                processed_data, 1, self.special_tokens
            )

            bigram_counts = count_n_grams(
                processed_data, 2, self.special_tokens
            )

            model = NGramModel(
                unigram_counts,
                bigram_counts,
                self.vocabulary,
                self.special_tokens,
                k=1
            )

            self.models.append(model)

            token_count = sum(len(s) for s in processed_data)
            self.style_token_counts.append(token_count)

        # Equal prior probabilities for all styles
        num_styles = len(self.models)

        self.style_priors = [
            1.0 / num_styles
            for _ in range(num_styles)
        ] if num_styles > 0 else []

    def write_in_style_ngram(self, passage, k=10):
        """
        Identify the most likely writing style and predict
        the top-k next words using its bigram language model.

        Returns:
            words: list of predicted words
            probabilities: list of corresponding probabilities
            style_file: index of the selected style
            probability_style: posterior probability of that style
        """

        if not self.models:
            return [], [], None, 0.0

        # Tokenize the input passage
        tokens = nltk.word_tokenize(passage.lower())

        # Replace OOV words using the shared vocabulary
        tokens = [
            word if word in self.vocab_set
            else self.special_tokens.unknown_token
            for word in tokens
        ]

        # Calculate log-likelihood for each style
        log_scores = []

        for index, model in enumerate(self.models):

            log_probability = math.log(
                self.style_priors[index]
            )

            previous_word = self.special_tokens.start_token

            for word in tokens:

                context_count = model.n_gram_counts.get(
                    (previous_word,), 0
                )

                bigram_count = model.n_plus1_gram_counts.get(
                    (previous_word, word), 0
                )

                vocab_size = len(self.vocabulary) + 2

                probability = (
                    bigram_count + model.k
                ) / (
                    context_count + model.k * vocab_size
                )

                log_probability += math.log(probability)

                previous_word = word

            log_scores.append(log_probability)

        # Convert log-scores to posterior probabilities
        # using the log-sum-exp technique
        max_score = max(log_scores)

        exp_scores = [
            math.exp(score - max_score)
            for score in log_scores
        ]

        total = sum(exp_scores)

        style_probabilities = [
            score / total
            for score in exp_scores
        ]

        # Select the most likely writing style
        style_file = max(
            range(len(style_probabilities)),
            key=lambda i: style_probabilities[i]
        )

        probability_style = style_probabilities[style_file]

        selected_model = self.models[style_file]

        # Bigram prediction uses the last token as context
        if tokens:
            context_tokens = [tokens[-1]]
        else:
            context_tokens = [self.special_tokens.start_token]

        probabilities_dict = estimate_probabilities(
            context_tokens,
            selected_model
        )

        # Sort candidates by descending probability
        sorted_predictions = sorted(
            probabilities_dict.items(),
            key=lambda item: item[1],
            reverse=True
        )

        # Select top-k candidates
        top_predictions = sorted_predictions[:max(0, k)]

        words = [
            word for word, probability in top_predictions
        ]

        probabilities = [
            probability for word, probability in top_predictions
        ]

        return (
            words,
            probabilities,
            style_file,
            probability_style
        )

In [2]:
stylegram = StyleGram(style_files)

words, probabilities, style_index, style_probability = (
    stylegram.write_in_style_ngram(
        "The sun was shining",
        k=10
    )
)

print("Predicted words:", words)
print("Word probabilities:", probabilities)
print("Style index:", style_index)
print("Style probability:", style_probability)

NameError: name 'style_files' is not defined